# ch06 — Gen5: MambaTSAD faithful vs fixed (v2)

Issues verified in the official code (aaaceo890/mamba_tsad @ 08fec05): (a) inference state indexing, (c) HP objective, (d) batch-global AMA period. Small CPU settings — illustrative only; paper settings need a GPU (configs/mamba_paper_repro.yaml).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tsad_forge.synthetic.generator import generate_synthetic

In [ ]:
from tsad_forge.models.registry import get_model
from tsad_forge.evaluation.metrics import compute_metrics
from tsad_forge.evaluation.protocol import zscore_normalize

ds = generate_synthetic(n_events=6, seed=6)
train, test = zscore_normalize(ds.train, ds.test)
for name in ["mamba_tsad_faithful", "mamba_tsad_fixed"]:
    vals = []
    for seed in range(3):
        s = get_model(name, seed=seed, epochs=2, window=32, d_model=16).fit(train).score(test)
        vals.append(compute_metrics(s, ds.labels)["vus_pr"])
    print(f"{name:22s} VUS-PR = {np.mean(vals):.3f} +/- {np.std(vals):.3f} (3 seeds)")

In [ ]:
# Issue (c): the official HP objective adds a first-difference term 2λD1'D1.
# Both variants feed the cycle (x - trend) to the model; only the trend differs.
from tsad_forge.models.gen5_ssm_foundation.mamba_tsad import hp_filter
y = ds.test[:300, :1]
trend_std, _ = hp_filter(y, lam=1e4, objective="standard")
trend_up, _ = hp_filter(y, lam=1e4, objective="upstream")
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(y[:, 0], lw=0.6, color="0.6", label="original")
ax.plot(trend_std[:, 0], lw=1.4, label="trend — standard HP (fixed)")
ax.plot(trend_up[:, 0], lw=1.4, ls="--", label="trend — official objective (faithful)")
ax.legend(loc="upper right")
plt.tight_layout()